In [1]:
from model_ranking import (
    mutual_information_from_probs_torch,
    mutual_information_from_probs_np,
    hopkins_statistic,
    uniformity,
    load_h5,
    FeatureExtractor,
    calculate_transfer_metric,
    ensure_even_feature_sampling,
    PrecomputedFeatureConfig,
    PrecomputedDirectPerformanceConfig,
)

from model_ranking.transferability_metrics.transfer_metrics import (
    get_transfer_data_segmentation,
)

from pytorch3dunet.unet3d.model import UNet2D
from typing import Dict, Any
import torch
import numpy as np
import math


INFO: P [MainThread] 2025-11-05 11:00:03,263 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EtoE_feature_path = "/g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/EPFL_to_EPFL/E_model5_to_EPFL_features.h5"
layer_name = "decoders.2"
features_per_patch = load_h5(EtoE_feature_path, f"{layer_name}_features")
labels_per_patch = load_h5(EtoE_feature_path, f"{layer_name}_labels")
predictions_per_patch = load_h5(EtoE_feature_path, f"{layer_name}_predictions")

In [3]:
feature_cfg = PrecomputedFeatureConfig(
    base_path="/g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled",
    file_type="h5",
    layer_keys = {
        "NA": "decoders.2",
        "Res": "decoders.3",
        "Unetr": "decoder2",
    },
    n_PCA_components=None
)

performance_cfg = PrecomputedDirectPerformanceConfig(
    name="direct_performance",
    base_path="/scratch/talks/consistency_results/patch_segmentation/mitochondria",
    approach="consistency", 
    run_id="P_full",
    key="hard_f1"
)

(
    features,
    predictions,
    labels,
    performance_score,
) = get_transfer_data_segmentation(
    model_name = "E_model5",
    target= "EPFL",
    epoch="",
    feature_config=feature_cfg,
    transferability_metric="Transfer_Score",
    performance_config=performance_cfg,
)

In [6]:
unique, counts = np.unique(labels, return_counts=True)
dict(zip(unique, counts))

{0: 389666, 1: 322334}

In [7]:
even_features, even_labels, even_preds = ensure_even_feature_sampling(labels, features, predictions)

Original dataset: 712000 samples
Balanced dataset: 644668 samples


In [10]:
import torch

a = torch.empty((1, 32))
print(a.shape)

torch.Size([1, 32])


# Nuclei

In [10]:
feature_cfg = PrecomputedFeatureConfig(
    base_path="/g/kreshuk/talks/sampled_features/semantic_segmentation/nuclei/1k_pixels_sampled",
    file_type="h5",
    layer_keys = {
        "BC": "decoders.1",
        "HN": "decoders.1",
        "Hst": "decoders.1",
        "895_model2": "decoders.1",
        "GN": "decoders.2",
        "1196_model3": "decoders.2",
        "1410_model2": "decoders.2",
    },
    n_PCA_components=None
)

performance_cfg = PrecomputedDirectPerformanceConfig(
    name="direct_performance",
    base_path="/g/kreshuk/talks/consistency_results/patch_segmentation/nuclei",
    approach="consistency", 
    run_id="P_1",
    key="hard_f1",
    metric_summary=True,
)

(
    features,
    predictions,
    labels,
    performance_score,
) = get_transfer_data_segmentation(
    model_name = "BC_model4",
    target= "BBBC039",
    epoch="",
    feature_config=feature_cfg,
    transferability_metric="Transfer_Score",
    performance_config=performance_cfg,
)

In [11]:
# BBBC039
unique, counts = np.unique(labels, return_counts=True)
dict(zip(unique, counts))

{0: 24570, 1: 23430}

In [7]:
# DSB2018
unique, counts = np.unique(labels, return_counts=True)
dict(zip(unique, counts))

{0: 28697, 1: 21303}

In [9]:
# S_BIAD895
unique, counts = np.unique(labels, return_counts=True)
dict(zip(unique, counts))

{0: 23832, 1: 21168}

In [ ]:
# Hoechst
unique, counts = np.unique(labels, return_counts=True)
dict(zip(unique, counts))

{0: 5195, 1: 4805}